# Modélisation initiale — ATD + Smart Tagging

Notebook correspondant aux étapes 1 à 3 de `plan_modelisation_possession_mtaH2027.md`.

Il ne refait pas la baseline et n'utilise pas de HMM. Il charge les données tactiques, contrôle la timeline, construit des features ATD invariantes aux slots et transforme le Smart Tagging en variables frame-level.

In [ ]:
import os, re, json, xml.etree.ElementTree as ET
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

FPS, N_FRAMES, PITCH_L, PITCH_W = 25, 199319, 105.3, 68.0
project_root = Path.cwd()
if not (project_root / 'data').is_dir() and (project_root.parent / 'data').is_dir():
    project_root = project_root.parent
data_root = project_root / 'data'
if not (data_root / 'tactical').is_dir() or not (data_root / 'broadcast').is_dir():
    raise FileNotFoundError('Dossier local data/tactical et data/broadcast introuvable.')
tactical = data_root / 'tactical'
assets = {k: tactical/n for k,n in {
    'meta':'BAR-ATH_tactical_atd.xml', 'atd':'BAR-ATH_tactical_atd.txt',
    'homography':'BAR-ATH_tactical_homography.csv',
    'tags':'BAR-ATH_tactical_smart_tagging.xml', 'video':'BAR-ATH_tactical.mp4'}.items()}
for key, path in assets.items(): print(f'{key:12s} {path} [{"OK" if path.exists() else "ABSENT"}]')

In [ ]:
def metadata(path):
    root = ET.parse(path).getroot()
    teams = [(x.get('id'), x.findtext('Name')) for x in root.iter('Team')]
    home_id = teams[0][0]
    slot_team = np.array([0 if x.get('teamId') == home_id else 1 for x in root.iter('Player')], dtype=np.int8)
    return teams, slot_team

teams, slot_team = metadata(assets['meta'])
print('H:', teams[0][1], '| A:', teams[1][1], '| slots:', len(slot_team))

def triplet(value):
    if not value or value.startswith('N'): return (np.nan, np.nan, np.nan)
    out = []
    for x in value.split(',')[:3]:
        try: out.append(float(x))
        except ValueError: out.append(np.nan)
    return tuple((out + [np.nan]*3)[:3])

def load_atd(path, n_slots):
    ids, player, ball = [], [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(':')
            if len(parts) != 3: continue
            ids.append(int(parts[0]))
            row = np.full((n_slots, 3), np.nan, dtype=np.float32)
            for i, value in enumerate(parts[1].split(';')[:n_slots]): row[i] = triplet(value)
            player.append(row); ball.append(triplet(parts[2]))
    return np.asarray(ids), np.asarray(player), np.asarray(ball)

frame_id, player, ball = load_atd(assets['atd'], len(slot_team))
assert len(frame_id) == N_FRAMES and np.array_equal(frame_id, np.arange(1, N_FRAMES+1))
assert player.shape == (N_FRAMES, len(slot_team), 3)
print('ATD:', player.shape, '| video frames:', frame_id[0]-1, '->', frame_id[-1]-1)

In [ ]:
# QA timeline et missingness, selon DATA.md.
ball_visible = np.isfinite(ball[:, :2]).all(axis=1)
player_visible = np.isfinite(player[:, :, :2]).all(axis=2)
graded = np.zeros(N_FRAMES, dtype=bool); graded[11185:90652] = True; graded[110808:190198] = True
def gap_before(mask):
    out, n = np.zeros(len(mask), dtype=np.int32), 0
    for i, x in enumerate(mask): n = n+1 if not x else 0; out[i] = n
    return out
qa = {
 'n_frames': int(len(frame_id)), 'ball_visible_total': float(ball_visible.mean()),
 'ball_visible_graded': float(ball_visible[graded].mean()),
 'players_mean_graded': float(player_visible[graded].sum(1).mean()),
 'players_min_graded': int(player_visible[graded].sum(1).min()),
 'players_max_graded': int(player_visible[graded].sum(1).max())}
print(json.dumps(qa, indent=2))
plt.figure(figsize=(14,3)); plt.plot(gap_before(ball_visible)/FPS, linewidth=.25); plt.ylabel('Ball gap (s)'); plt.xlabel('frame'); plt.show()

## Features ATD

Les features sont collectives : aucun slot n'est traité comme une identité permanente. Les distances sont calculées en mètres, tandis que les coordonnées normalisées sont conservées pour les projections.

In [ ]:
p = player[:, :, :2].astype(np.float32).copy(); p[:,:,0] *= PITCH_L; p[:,:,1] *= PITCH_W
b = ball[:, :2].astype(np.float32).copy(); b[:,0] *= PITCH_L; b[:,1] *= PITCH_W
dist = np.sqrt(((p - b[:,None,:])**2).sum(2)); dist[~player_visible | ~ball_visible[:,None]] = np.nan
features = pd.DataFrame({'frame': np.arange(N_FRAMES), 'ball_visible': ball_visible.astype('int8'),
 'ball_x_norm': ball[:,0], 'ball_y_norm': ball[:,1], 'ball_x_m': b[:,0], 'ball_y_m': b[:,1],
 'players_visible_total': player_visible.sum(1), 'ball_gap_before_frames': gap_before(ball_visible),
 'is_graded_frame': graded.astype('int8')})
for code, name in ((0,'home'), (1,'away')):
    mask = slot_team == code; d = dist[:,mask]; pos = p[:,mask,:]; visible = player_visible[:,mask]
    order = np.sort(np.where(np.isfinite(d), d, np.inf), axis=1); order[order == np.inf] = np.nan
    for rank in range(3): features[f'{name}_nearest_{rank+1}_m'] = order[:,rank]
    for radius in (1,2,5,10): features[f'{name}_within_{radius}m'] = np.nansum(d <= radius, axis=1)
    features[f'{name}_players_visible'] = visible.sum(1)
    features[f'{name}_centroid_x_m'] = np.nanmean(pos[:,:,0], axis=1)
    features[f'{name}_centroid_y_m'] = np.nanmean(pos[:,:,1], axis=1)
    features[f'{name}_spread_x_m'] = np.nanstd(pos[:,:,0], axis=1)
    features[f'{name}_spread_y_m'] = np.nanstd(pos[:,:,1], axis=1)
features['nearest_distance_diff_m'] = features.home_nearest_1_m - features.away_nearest_1_m
features['nearest_distance_gap_m'] = features.nearest_distance_diff_m.abs()
features['centroid_distance_m'] = np.hypot(features.home_centroid_x_m-features.away_centroid_x_m, features.home_centroid_y_m-features.away_centroid_y_m)
for c in ('ball_x_norm','ball_y_norm'):
    interp = features[c].interpolate(limit=5, limit_area='inside')
    features[c.replace('_norm','_short_interp_norm')] = interp
    features[c.replace('_norm','_short_interp_used')] = (features.ball_visible.eq(0) & interp.notna()).astype('int8')
features['ball_gap_before_s'] = features.ball_gap_before_frames / FPS
features['tracking_complete_enough'] = ((features.home_players_visible >= 8) & (features.away_players_visible >= 8)).astype('int8')
print(features.shape); display(features.head())

## Smart Tagging frame-level

Les tags sont des indices contextuels. Le champ `Team` peut alimenter un prior, mais ne devient jamais directement le label de possession.

In [ ]:
def load_tags(path):
    out=[]
    for x in ET.parse(path).iter('instance'):
        start, end = float(x.findtext('start')), float(x.findtext('end'))
        out.append({'code': (x.findtext('code') or '').strip().upper(), 'start_s': start, 'end_s': end,
                    'start': max(0,int(round(start*FPS))), 'end': min(N_FRAMES,int(round(end*FPS))),
                    'tags': {z.findtext('group'): z.findtext('text') for z in x.findall('label')}})
    return out
events = load_tags(assets['tags']); print('events:', len(events))
codes = sorted({x['code'] for x in events})
for code in codes:
    key = 'tag_' + re.sub('[^A-Z0-9]+','_',code).strip('_').lower(); features[key] = 0
features['tag_active_count']=0; features['tag_team_score']=0.0; features['tag_set_piece']=0; features['tag_transition']=0
features['tag_seconds_since_start']=np.nan; features['tag_seconds_to_end']=np.nan
for event in events:
    a, z = event['start'], event['end']
    if z <= a: continue
    key = 'tag_' + re.sub('[^A-Z0-9]+','_',event['code']).strip('_').lower()
    features.loc[a:z-1,key] = 1; features.loc[a:z-1,'tag_active_count'] += 1
    team = str(event['tags'].get('Team','')).strip().lower()
    if team == 'barcelona': features.loc[a:z-1,'tag_team_score'] += 1
    if team in {'atletic club','athletic club'}: features.loc[a:z-1,'tag_team_score'] -= 1
    if event['code'] == 'SET PIECES': features.loc[a:z-1,'tag_set_piece'] = 1
    if 'TRANSITION' in event['code']: features.loc[a:z-1,'tag_transition'] = 1
    features.loc[a:z-1,'tag_seconds_since_start'] = features.loc[a:z-1,'frame']/FPS-event['start_s']
    features.loc[a:z-1,'tag_seconds_to_end'] = event['end_s']-features.loc[a:z-1,'frame']/FPS
features['tag_absent'] = (features.tag_active_count == 0).astype('int8')
print(features.filter(regex='^(frame|tag_)').head()); print(pd.Series([x['tags'].get('Team') for x in events]).value_counts(dropna=False))

In [ ]:
# Homographie : contrôles et features de confiance.
homography = pd.read_csv(assets['homography'])
matrix_cols = [f'h{r}{c}' for r in range(3) for c in range(3)]
assert len(homography) == N_FRAMES and np.array_equal(homography.frame.to_numpy(), np.arange(N_FRAMES))
assert set(matrix_cols + ['reprojection_error','valid']).issubset(homography.columns)
features['homography_valid'] = homography.valid.astype('int8')
features['homography_reprojection_error'] = homography.reprojection_error.astype('float32')
qa.update({'homography_valid': float(homography.valid.mean()), 'smart_tagging_events': len(events),
           'feature_count': len(features.columns), 'missing_rate_mean': float(features.isna().mean().mean())})
print(json.dumps(qa, indent=2, ensure_ascii=False))

# Export local : ne jamais versionner ces dérivés du match.
out = tactical.parent / 'derived_features'; out.mkdir(exist_ok=True)
features.to_parquet(out/'BAR-ATH_tactical_features_initiales.parquet', index=False)
with open(out/'BAR-ATH_tactical_qa.json','w',encoding='utf-8') as f: json.dump(qa,f,indent=2,ensure_ascii=False)
print('Export:', out)
display(features.isna().mean().sort_values(ascending=False).head(15).to_frame('missing_rate'))

## Baselines officielles : résultat de référence

Cette section appelle le script fourni dans `starter/baseline.py`. Elle ne réimplémente pas la baseline. Le ground truth public sert uniquement à vérifier le pipeline et à visualiser la référence sur les cinq premières minutes annotées.

In [ ]:
import sys
sys.path.insert(0, str(project_root))
from starter.baseline import build as official_baseline, smooth as official_smooth
from evaluation import possession_metrics as pm

baseline_codes = official_baseline(data_dir=str(data_root))
baseline_windows = {'raw': 0, 'smooth25': 25, 'smooth50': 50, 'smooth125': 125}
baseline_labels = {name: official_smooth(baseline_codes, window) for name, window in baseline_windows.items()}

gt_path = project_root / 'evaluation/gt/BAR-ATH_possession_gt_public.csv'
if not gt_path.exists():
    raise FileNotFoundError('Ground truth public introuvable : lance le notebook depuis la racine du dépôt.')
gt_codes = pm.label_codes(pm.read_label_csv(gt_path))
baseline_metrics = []
for name, labels in baseline_labels.items():
    result = pm.evaluate(gt_codes, pm.label_codes(labels))
    baseline_metrics.append({'baseline': name, 'window_frames': baseline_windows[name],
                            'macro_f1': result['macro_f1'], 'spell_f1': result['spell_f1_mean'],
                            'n_pred_spells': result['n_pred_spells'],
                            'mean_spell_duration_s': result['mean_spell_duration_s']})
baseline_table = pd.DataFrame(baseline_metrics).sort_values('spell_f1', ascending=False)
display(baseline_table.style.format({'macro_f1':'{:.4f}', 'spell_f1':'{:.4f}', 'mean_spell_duration_s':'{:.2f}'}))

In [ ]:
# Visualisation sur la fenêtre publique : 11 185 -> 18 684.
public_start, public_end = 11185, 18685
label_to_number = {'D': 0, 'A': 1, 'H': 2, 'X': -1}
fig, axes = plt.subplots(2, 1, figsize=(16, 7), sharex=True, constrained_layout=True)
for name, labels in baseline_labels.items():
    values = np.array([label_to_number[x] for x in labels[public_start:public_end]])
    axes[0].step(np.arange(public_start, public_end), values, where='post', label=name, linewidth=.8)
axes[0].set_yticks([0,1,2], ['D','A','H']); axes[0].set_ylabel('Label prédit'); axes[0].legend(ncol=4)
axes[0].set_title('Baselines officielles — fenêtre publique')
axes[1].bar(baseline_table['baseline'], baseline_table['spell_f1'], color='#4472C4')
axes[1].axhline(.2433, color='black', linestyle='--', label='référence complète smooth50 = 0.2433')
axes[1].set_ylabel('spellF1 public'); axes[1].legend()
plt.show()

## Signaux de la modélisation

Ces graphiques ne sélectionnent aucun seuil. Ils servent à comprendre les signaux disponibles : ballon observé ou absent, compétition des distances H/A, contexte Smart Tagging et qualité du tracking.

In [ ]:
sample_start, sample_end = 11185, 18685
sample = features.iloc[sample_start:sample_end]
fig, axes = plt.subplots(4, 1, figsize=(16, 10), sharex=True, constrained_layout=True)
axes[0].plot(sample.frame, sample.ball_visible, linewidth=.7); axes[0].set_ylabel('Ballon visible')
axes[1].plot(sample.frame, sample.nearest_distance_diff_m, label='H - A', linewidth=.7)
axes[1].axhline(0, color='black', linewidth=.5); axes[1].set_ylabel('Distance proche H-A (m)')
axes[2].plot(sample.frame, sample.tag_team_score, label='prior équipe tags', linewidth=.7, color='#ED7D31')
axes[2].plot(sample.frame, sample.tag_active_count, label='tags actifs', linewidth=.5, color='#70AD47')
axes[2].legend(loc='upper right'); axes[2].set_ylabel('Smart Tagging')
axes[3].plot(sample.frame, sample.home_players_visible, label='H', linewidth=.7)
axes[3].plot(sample.frame, sample.away_players_visible, label='A', linewidth=.7)
axes[3].set_ylabel('Tracks visibles'); axes[3].set_xlabel('Frame tactique'); axes[3].legend()
for ax in axes:
    for event in events:
        if event['end'] > sample_start and event['start'] < sample_end:
            ax.axvspan(max(event['start'],sample_start), min(event['end'],sample_end), color='grey', alpha=.04)
plt.show()

In [ ]:
# Vue spatiale d'une frame : les agrégats collectifs remplacent une identité de joueur fragile.
snapshot = int(features.loc[features.ball_visible.eq(1) & features.is_graded_frame.eq(1), 'frame'].iloc[len(features)//20])
fig, ax = plt.subplots(figsize=(12, 7))
for code, color, name in ((0, '#4472C4', 'Barcelona'), (1, '#ED7D31', 'Athletic')):
    visible = np.isfinite(player[snapshot,:,0]) & np.isfinite(player[snapshot,:,1]) & (slot_team == code)
    ax.scatter(player[snapshot,visible,0]*PITCH_L, player[snapshot,visible,1]*PITCH_W, s=35, color=color, label=name)
ax.scatter(ball[snapshot,0]*PITCH_L, ball[snapshot,1]*PITCH_W, s=90, color='black', marker='*', label='ballon')
ax.set_xlim(0, PITCH_L); ax.set_ylim(0, PITCH_W); ax.set_aspect('equal')
ax.set_xlabel('X (m)'); ax.set_ylabel('Y (m)'); ax.set_title(f'Frame {snapshot} — positions ATD')
ax.legend(); ax.grid(alpha=.2); plt.show()
print('Features à cette frame :')
display(features.loc[[snapshot], ['ball_visible','home_nearest_1_m','away_nearest_1_m','nearest_distance_diff_m','tag_team_score','home_players_visible','away_players_visible']])

## Suite du plan

Ce notebook prépare les features et montre les références existantes. Il ne règle aucun hyperparamètre sur les 17 possessions publiques. La prochaine étape est l'ablation ATD / ATD+Smart Tagging sur un corpus externe annoté, puis l'entraînement d'un modèle temporel sans HMM si cette voie est retenue.